In [ ]:
from manim import *
import os

class SelfAttentionMechanism(Scene):
    def construct(self):
        # ==========================================
        # 0. SETUP & HELPERS
        # ==========================================
        self.camera.background_color = "#1e1e1e"
        
        # Helper: Create Matrix with Label
        def get_matrix(data, label, color, scale=0.6):
            m = Matrix(data, v_buff=0.5, h_buff=0.7, bracket_h_buff=0.1).set_column_colors(color)
            m.scale(scale)
            lbl = MathTex(label, color=color).scale(scale).next_to(m, UP, buff=0.1)
            return VGroup(lbl, m)
            
        # Helper: Highlight Code Line (The "Focus" Mechanism)
        # This robustly finds the line of text regardless of Manim version
        def highlight_line(line_index):
            lines = VGroup()
            
            # 1. Try finding lines via attribute (Manim Community standard)
            if hasattr(code_obj, 'code'):
                lines = code_obj.code
            # 2. Fallback: Try index 2 (Common in [Background, LineNos, Code] structure)
            elif len(code_obj) >= 3:
                lines = code_obj[2]
            # 3. Fallback: Try index 1 (Minimal structure)
            elif len(code_obj) >= 2:
                lines = code_obj[1]
            
            # Validation
            if line_index < 0 or line_index >= len(lines): 
                return VGroup()
            
            target_line = lines[line_index]
            
            # Create the Glow/Rectangle
            rect = SurroundingRectangle(target_line, color=YELLOW, stroke_width=2, buff=0.05)
            rect.stretch(1.1, 0) # Make it slightly wider than the text
            
            # Add a small label on the right for clarity (Optional, helps focus)
            arrow = Arrow(LEFT, RIGHT, color=YELLOW).next_to(rect, LEFT, buff=0.1).scale(0.5)
            
            return VGroup(rect)

        # ==========================================
        # 1. GENERATE CODE
        # ==========================================
        source_content = """import torch
import torch.nn.functional as F
import math

# Input Embeddings
X = Embeddings(["Data", "Flow"])

# 1. Projections (Q, K, V)
Q = X @ W_q
K = X @ W_k
V = X @ W_v

# 2. Scaled Dot-Product
scores = (Q @ K.T) / math.sqrt(d_k)

# 3. Softmax
attn = F.softmax(scores, dim=-1)

# 4. Output Aggregation
Z = attn @ V
"""
        filename = "attention_safe.py"
        with open(filename, "w") as f:
            f.write(source_content)

        # ==========================================
        # 2. LAYOUT SETUP
        # ==========================================
        divider = Line(UP*4, DOWN*4, color=GRAY_D).shift(LEFT*2)
        
        # Code Section (Left)
        # SAFE INITIALIZATION: Passing only positional filename + language
        try:
            code_obj = Code(filename, language="python")
        except:
            # Absolute fallback for very old versions
            code_obj = Code(filename)

        code_obj.scale(0.5).next_to(divider, LEFT, aligned_edge=RIGHT).shift(LEFT*0.2)
        
        # Cleanup: Hide line numbers manually to avoid kwarg errors
        if len(code_obj) > 2:
            code_obj[1].set_opacity(0) # Index 1 is usually line numbers
        
        title_code = Text("PyTorch Source", font_size=20, weight=BOLD, color=GRAY_B)
        title_code.next_to(code_obj, UP, aligned_edge=LEFT)
        
        # Formula (Top Right)
        formula = MathTex(
            r"\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V",
            font_size=24
        ).to_corner(UR).shift(LEFT*0.5)

        # Explanation Area (Bottom Right)
        RIGHT_CENTER = RIGHT * 3.5
        expl_text = Text("Initializing...", font_size=20, color=WHITE).to_edge(DOWN, buff=0.5).set_x(RIGHT_CENTER[0])

        self.add(divider, code_obj, title_code, formula, expl_text)

        # ==========================================
        # 3. ANIMATION LOGIC
        # ==========================================
        curr_hl = VGroup() # Tracks the current highlight object

        def update_explanation(text):
            new_expl = Text(text, font_size=20, color=YELLOW).move_to(expl_text.get_center())
            if new_expl.width > 6.5:
                new_expl.scale_to_fit_width(6.5)
            return Transform(expl_text, new_expl)

        # --- STEP 1: INPUT (Line 5) ---
        # Focus: X = Embeddings...
        new_hl = highlight_line(5) 
        self.play(Create(new_hl), update_explanation("Input X: Embeddings for 'Data' and 'Flow'"))
        curr_hl = new_hl

        x_vals = [[1, 0, 1], [0, 1, 0]]
        mat_X = get_matrix(x_vals, "X", WHITE).move_to(RIGHT_CENTER + UP*2)
        
        x_labels = VGroup(
            Text("Data", font_size=14).next_to(mat_X[1].get_rows()[0], LEFT, buff=0.1),
            Text("Flow", font_size=14).next_to(mat_X[1].get_rows()[1], LEFT, buff=0.1)
        )
        group_X = VGroup(mat_X, x_labels)
        
        self.play(FadeIn(group_X))
        self.wait(1)

        # --- STEP 2: PROJECTIONS (Lines 8, 9, 10) ---
        pos_q = RIGHT_CENTER + LEFT*2 + UP*0.5
        pos_k = RIGHT_CENTER + UP*0.5
        pos_v = RIGHT_CENTER + RIGHT*2 + UP*0.5
        
        projections = [
            ("Q", "W_q", BLUE, 8, pos_q, "Query: What am I looking for?"),
            ("K", "W_k", YELLOW, 9, pos_k, "Key: What do I contain?"),
            ("V", "W_v", GREEN, 10, pos_v, "Value: My actual content")
        ]
        
        stored_matrices = {}

        for name, w_name, color, line_idx, final_pos, meaning in projections:
            # 1. MOVE THE HIGHLIGHT RECTANGLE
            target_hl = highlight_line(line_idx)
            self.play(
                ReplacementTransform(curr_hl, target_hl), # Smooth glide animation
                update_explanation(f"{name} = X @ {w_name} ({meaning})")
            )
            curr_hl = target_hl
            
            # 2. Perform Math Animation
            calc_pos = RIGHT_CENTER + DOWN * 1.0
            
            x_copy = mat_X.copy().set_color(color).scale(0.8)
            
            # Robust copy handling
            matrix_part = x_copy
            if len(x_copy) > 1:
                matrix_part = x_copy[1]
                x_copy.remove(x_copy[0])
            
            self.add(matrix_part)
            
            self.play(matrix_part.animate.move_to(calc_pos + LEFT*1.5), run_time=0.5)
            
            w_mat = get_matrix([[1,0],[0,1],[1,1]], w_name, color, scale=0.4)
            w_mat.next_to(matrix_part, RIGHT, buff=0.2)
            self.play(FadeIn(w_mat), run_time=0.5)
            
            dot = MathTex(r"\times").move_to((matrix_part.get_center() + w_mat.get_center())/2)
            self.add(dot)
            self.wait(0.3)
            
            res_vals = [[1, 1], [0, 1]] if name == "Q" else [[0, 1], [1, 0]]
            res_mat = get_matrix(res_vals, name, color, scale=0.5)
            res_mat.move_to(calc_pos + RIGHT*1.0)
            
            self.play(
                ReplacementTransform(VGroup(matrix_part, w_mat, dot), res_mat),
                run_time=0.8
            )
            
            self.play(res_mat.animate.move_to(final_pos), run_time=0.6)
            stored_matrices[name] = res_mat

        self.play(FadeOut(group_X))

        # --- STEP 3: SCORES (Line 13) ---
        target_hl = highlight_line(13) # scores = ...
        self.play(
            ReplacementTransform(curr_hl, target_hl),
            update_explanation("Scores = Q @ K.T (Similarity)")
        )
        curr_hl = target_hl

        q_mat = stored_matrices["Q"]
        k_mat = stored_matrices["K"]
        v_mat = stored_matrices["V"]

        self.play(v_mat.animate.scale(0.8).to_corner(DR).shift(UP*1.5 + LEFT*0.5))

        calc_center = RIGHT_CENTER + DOWN*0.5
        self.play(
            q_mat.animate.move_to(calc_center + LEFT*1.2),
            k_mat.animate.move_to(calc_center + RIGHT*1.2)
        )

        k_t = get_matrix([[0, 1], [1, 0]], "K^T", YELLOW, scale=0.5).move_to(k_mat.get_center())
        trans_lbl = Text("Transpose", font_size=12, color=YELLOW).next_to(k_t, UP, buff=0)
        self.play(Transform(k_mat, k_t), FadeIn(trans_lbl))
        self.wait(0.5)
        self.play(FadeOut(trans_lbl))

        dot_sym = MathTex(r"\cdot").scale(1.5).move_to(calc_center)
        self.play(FadeIn(dot_sym))
        
        scores = get_matrix([[2, 1], [1, 3]], "Scores", RED, scale=0.6).move_to(calc_center)
        self.play(ReplacementTransform(VGroup(q_mat, k_mat, dot_sym), scores))

        self.play(update_explanation("Scale by sqrt(d_k) to stabilize gradients"))
        denom = MathTex(r"/ \sqrt{d_k}").next_to(scores, RIGHT)
        self.play(Write(denom))
        self.wait(0.5)
        
        scaled_scores = get_matrix([[1.4, 0.7], [0.7, 2.1]], "Scaled", RED, scale=0.6).move_to(scores.get_center())
        self.play(Transform(scores, scaled_scores), FadeOut(denom))

        # --- STEP 4: SOFTMAX (Line 16) ---
        target_hl = highlight_line(16) # attn = F.softmax...
        self.play(
            ReplacementTransform(curr_hl, target_hl),
            update_explanation("Softmax: Normalize to Probabilities")
        )
        curr_hl = target_hl

        self.play(scores.animate.shift(UP*0.8))
        
        arrow = Arrow(scores.get_bottom(), scores.get_bottom() + DOWN*1.2, color=RED)
        attn = get_matrix([[0.7, 0.3], [0.2, 0.8]], "Attn Weights", RED, scale=0.6).next_to(arrow, DOWN)
        
        self.play(GrowArrow(arrow), FadeIn(attn))
        self.wait(1)
        
        self.play(
            FadeOut(scores), FadeOut(arrow),
            attn.animate.move_to(calc_center + UP*0.5)
        )

        # --- STEP 5: OUTPUT (Line 19) ---
        target_hl = highlight_line(19) # Z = attn @ V
        self.play(
            ReplacementTransform(curr_hl, target_hl),
            update_explanation("Output = Attn @ V (Mix Content)")
        )
        curr_hl = target_hl

        self.play(v_mat.animate.next_to(attn, RIGHT, buff=0.5).scale(1.25)) 
        
        dot_final = MathTex(r"\cdot").scale(1.5).move_to(midpoint(attn.get_center(), v_mat.get_center()))
        self.play(FadeIn(dot_final))
        
        z_out = get_matrix([[0.8, 0.9], [0.3, 0.8]], "Z (Output)", PURPLE, scale=0.7)
        z_out.move_to(calc_center + DOWN*1.5)
        
        self.play(ReplacementTransform(VGroup(attn, v_mat, dot_final), z_out))
        
        box = SurroundingRectangle(z_out, color=PURPLE, buff=0.2)
        self.play(Create(box))
        
        self.play(update_explanation("Contextualized embeddings ready."))
        self.wait(2)
        
        if os.path.exists(filename):
            os.remove(filename)

%manim -qk -v warning SelfAttentionMechanism

Manim Community v0.19.0